In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No API key was found!')
elif not api_key.startswith('sk-proj-'):
    print('An API key was found, but it doesnt start with "sk-proj-", i.e, it is not an OpenAI API key')
elif api_key.strip() != api_key:
    print('An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them.')
else:
    print('The API key found and looks good so far!')


The API key found and looks good so far!


# LangChain

The book's example (pp. 305-309) is a project manager agent. It turns a request into a task, sets a priority (P0, P1, P2) from words like "urgent", and assigns a worker, filling in reasonable defaults for whatever the request leaves out. Almost nothing in the listing survives on the current stack:

| The book | Here |
|---|---|
| `create_react_agent` + `AgentExecutor` from `langchain.agents` | removed in LangChain 1.x; `create_agent` |
| `ConversationBufferMemory` from `langchain.memory` | removed; `InMemorySaver` as the agent's checkpointer, with one `thread_id` for both requests |
| `Tool(name=..., func=..., args_schema=...)` | `Tool` passes a single string, and two of these tools take two arguments; `StructuredTool.from_function` |
| `ChatPromptTemplate` with `{chat_history}`, `{agent_scratchpad}` | the same system text as `system_prompt`; the agent manages its own messages |
| `verbose=True` | the listing's own `DEBUG` lines show each action, and the reply is printed |
| `asyncio.run(run_simulation())` under `__main__` | `await run_simulation(...)` |
| — | `check_board`, which reads the task manager directly |

Notice what this agent prioritizes: each task, on its own, from a keyword. Nothing ever orders the tasks against one another, or reconsiders a priority when a new task arrives. The book's prose covers those criteria (urgency, importance, dependencies, resources, dynamic re-prioritization); the code covers the first.

The check at the end looks at the **board, not the reply**. The agent's reply is a description of what it did, and it can be wrong, as the Ollama section shows.

The next cell holds the task manager, the tools, the prompt and the simulation, with no model. The OpenAI cell and the Ollama cell each build the agent and run it, so either can run on its own.

In [3]:
# Book listing: Chapter 20, Hands-On Code Example, pp. 305-309.
# This cell holds the task manager, the tools, the prompt and the simulation, and no model. The
# OpenAI cell and the Ollama cell below each pass their model to build_pm_agent and run it.
from typing import Dict, Optional

from langchain.agents import create_agent  # book: create_react_agent + AgentExecutor
from langchain_core.tools import StructuredTool  # book: Tool, which takes one string
from langgraph.checkpoint.memory import InMemorySaver  # book: ConversationBufferMemory
from pydantic import BaseModel, Field

# --- 1. Task Management System ---

class Task(BaseModel):
    """Represents a single task in the system."""
    id: str
    description: str
    priority: Optional[str] = None  # P0, P1, P2
    assigned_to: Optional[str] = None # Name of the worker

class SuperSimpleTaskManager:
    """An efficient and robust in-memory task manager."""
    def __init__(self):
        # Use a dictionary for O(1) lookups, updates, and deletions.
        self.tasks: Dict[str, Task] = {}
        self.next_task_id = 1

    def create_task(self, description: str) -> Task:
        """Creates and stores a new task."""
        task_id = f"TASK-{self.next_task_id:03d}"
        new_task = Task(id=task_id, description=description)
        self.tasks[task_id] = new_task
        self.next_task_id += 1
        print(f"DEBUG: Task created - {task_id}: {description}")
        return new_task

    def update_task(self, task_id: str, **kwargs) -> Optional[Task]:
        """Safely updates a task using Pydantic's model_copy."""
        task = self.tasks.get(task_id)
        if task:
            # Use model_copy for type-safe updates.
            update_data = {k: v for k, v in kwargs.items() if v is not None}
            updated_task = task.model_copy(update=update_data)
            self.tasks[task_id] = updated_task
            print(f"DEBUG: Task {task_id} updated with {update_data}")
            return updated_task

        print(f"DEBUG: Task {task_id} not found for update.")
        return None

    def list_all_tasks(self) -> str:
        """Lists all tasks currently in the system."""
        if not self.tasks:
            return "No tasks in the system."

        task_strings = []
        for task in self.tasks.values():
            task_strings.append(
                f"ID: {task.id}, Desc: '{task.description}', "
                f"Priority: {task.priority or 'N/A'}, "
                f"Assigned To: {task.assigned_to or 'N/A'}"
            )
        return "Current Tasks:\n" + "\n".join(task_strings)

task_manager = SuperSimpleTaskManager()

# --- 2. Tools for the Project Manager Agent ---

# Use Pydantic models for tool arguments for better validation and clarity.
class CreateTaskArgs(BaseModel):
    description: str = Field(description="A detailed description of the task.")

class PriorityArgs(BaseModel):
    task_id: str = Field(description="The ID of the task to update, e.g., 'TASK-001'.")
    priority: str = Field(description="The priority to set. Must be one of: 'P0', 'P1', 'P2'.")

class AssignWorkerArgs(BaseModel):
    task_id: str = Field(description="The ID of the task to update, e.g., 'TASK-001'.")
    worker_name: str = Field(description="The name of the worker to assign the task to.")

def create_new_task_tool(description: str) -> str:
    """Creates a new project task with the given description."""
    task = task_manager.create_task(description)
    return f"Created task {task.id}: '{task.description}'."

def assign_priority_to_task_tool(task_id: str, priority: str) -> str:
    """Assigns a priority (P0, P1, P2) to a given task ID."""
    if priority not in ["P0", "P1", "P2"]:
        return "Invalid priority. Must be P0, P1, or P2."
    task = task_manager.update_task(task_id, priority=priority)
    return f"Assigned priority {priority} to task {task.id}." if task else f"Task {task_id} not found."

def assign_task_to_worker_tool(task_id: str, worker_name: str) -> str:
    """Assigns a task to a specific worker."""
    task = task_manager.update_task(task_id, assigned_to=worker_name)
    return f"Assigned task {task.id} to {worker_name}." if task else f"Task {task_id} not found."

# All tools the PM agent can use
pm_tools = [
    StructuredTool.from_function(
        name="create_new_task",
        func=create_new_task_tool,
        description="Use this first to create a new task and get its ID.",
        args_schema=CreateTaskArgs
    ),
    StructuredTool.from_function(
        name="assign_priority_to_task",
        func=assign_priority_to_task_tool,
        description="Use this to assign a priority to a task after it has been created.",
        args_schema=PriorityArgs
    ),
    StructuredTool.from_function(
        name="assign_task_to_worker",
        func=assign_task_to_worker_tool,
        description="Use this to assign a task to a specific worker after it has been created.",
        args_schema=AssignWorkerArgs
    ),
    StructuredTool.from_function(
        name="list_all_tasks",
        func=task_manager.list_all_tasks,
        description="Use this to list all current tasks and their status."
    ),
]

# --- 3. Project Manager Agent Definition ---

PM_SYSTEM_PROMPT = """You are a focused Project Manager LLM agent. Your goal is to manage project tasks efficiently.

    When you receive a new task request, follow these steps:
    1.  First, create the task with the given description using the `create_new_task` tool. You must do this first to get a `task_id`.
    2.  Next, analyze the user's request to see if a priority or an assignee is mentioned.
        - If a priority is mentioned (e.g., "urgent", "ASAP", "critical"), map it to P0. Use `assign_priority_to_task`.
        - If a worker is mentioned, use `assign_task_to_worker`.
    3.  If any information (priority, assignee) is missing, you must make a reasonable default assignment (e.g., assign P1 priority and assign to 'Worker A').
    4.  Once the task is fully processed, use `list_all_tasks` to show the final state.

    Available workers: 'Worker A', 'Worker B', 'Review Team'
    Priority levels: P0 (highest), P1 (medium), P2 (lowest)
    """


def build_pm_agent(llm, middleware: tuple = ()):
    """Create the agent. The checkpointer keeps the conversation between the two requests."""
    return create_agent(llm, pm_tools, system_prompt=PM_SYSTEM_PROMPT, middleware=middleware, checkpointer=InMemorySaver())


# --- 4. Simple Interaction Flow ---

async def ask_pm(pm_agent, request: str) -> None:
    """Sends one request on the shared thread and prints the agent's final reply."""
    config = {"configurable": {"thread_id": "project-1"}}
    result = await pm_agent.ainvoke({"messages": [("human", request)]}, config)
    print(f"--- {result['messages'][-1].text}")


# What the prompt asks for, checked on the task manager and not on the agent's reply.
AVAILABLE_WORKERS = ["Worker A", "Worker B", "Review Team"]


def check_board() -> None:
    """TASK-001 was stated exactly; TASK-002 only needs a reasonable default for both fields."""
    board = {task.id: (task.priority, task.assigned_to) for task in task_manager.tasks.values()}
    print(f"\nActual board: {board}")
    first = board.get("TASK-001") == ("P0", "Worker B")
    priority, worker = board.get("TASK-002", (None, None))
    second = priority in ("P0", "P1", "P2") and worker in AVAILABLE_WORKERS
    print(f"TASK-001 is P0 on Worker B: {first}")
    print(f"TASK-002 has a priority and an available worker: {second}")


async def run_simulation(pm_agent) -> None:
    print("--- Project Manager Simulation ---")
    task_manager.__init__()  # each model starts with an empty board

    # Scenario 1: Handle a new, urgent feature request
    print("\n[User Request] I need a new login system implemented ASAP. It should be assigned to Worker B.")
    await ask_pm(pm_agent, "Create a task to implement a new login system. It's urgent and should be assigned to Worker B.")

    print("\n" + "-"*60 + "\n")

    # Scenario 2: Handle a less urgent content update with fewer details
    print("[User Request] We need to review the marketing website content.")
    await ask_pm(pm_agent, "Manage a new task: Review marketing website content.")

    print("\n--- Simulation Complete ---")
    check_board()

In [4]:
# The book's project manager on OpenAI.
from langchain_openai import ChatOpenAI

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set; run the Ollama cell below instead.")

# The ChatOpenAI client automatically picks up the API key from the environment.
llm = ChatOpenAI(temperature=0.5, model="gpt-4o-mini")

await run_simulation(build_pm_agent(llm))

--- Project Manager Simulation ---

[User Request] I need a new login system implemented ASAP. It should be assigned to Worker B.


DEBUG: Task created - TASK-001: Implement a new login system.


DEBUG: Task TASK-001 updated with {'priority': 'P0'}DEBUG: Task TASK-001 updated with {'assigned_to': 'Worker B'}



--- The task to implement a new login system has been created with the following details:

- **Task ID**: TASK-001
- **Description**: Implement a new login system.
- **Priority**: P0 (Urgent)
- **Assigned To**: Worker B

If you need anything else, feel free to ask!

------------------------------------------------------------

[User Request] We need to review the marketing website content.


DEBUG: Task created - TASK-002: Review marketing website content.


DEBUG: Task TASK-002 updated with {'priority': 'P1'}DEBUG: Task TASK-002 updated with {'assigned_to': 'Worker A'}



--- The task to review marketing website content has been created with the following details:

- **Task ID**: TASK-002
- **Description**: Review marketing website content.
- **Priority**: P1 (Medium)
- **Assigned To**: Worker A

Here are all current tasks:

1. **Task ID**: TASK-001
   - **Description**: Implement a new login system.
   - **Priority**: P0 (Urgent)
   - **Assigned To**: Worker B

2. **Task ID**: TASK-002
   - **Description**: Review marketing website content.
   - **Priority**: P1 (Medium)
   - **Assigned To**: Worker A

If you have more tasks or need further assistance, let me know!

--- Simulation Complete ---

Actual board: {'TASK-001': ('P0', 'Worker B'), 'TASK-002': ('P1', 'Worker A')}
TASK-001 is P0 on Worker B: True
TASK-002 has a priority and an available worker: True


## LangChain + Ollama

This model has a habit the hosted one does not. It sends `create_new_task`, the two updates and `list_all_tasks` **all in one turn**, guessing the new task's ID. The agent runs a turn's tool calls concurrently (the hosted output above shows that too: its two updates print on one line), but `gpt-4o-mini` waits for the real ID before sending them. Here an update can arrive before the task exists: `Task TASK-002 not found for update`. In one such run the task ended with no priority and no owner, while the reply said it had "a default priority of P1" and was "assigned to the Review Team". It reported work that never happened.

Asking it to call one tool at a time worked in 3 runs of 5. So the cell takes the option away, the same idea as `disallow_transfer_to_*` in chapter 7: a small `after_model` middleware keeps only the first tool call of each turn. That removes the race, because only one call runs at a time, and the model sees each real ID before its next call.

That fixes the order, not the judgement. In 2 runs of 8 the model simply skipped setting the priority on the second task, and the board check caught it, so this cell can print `False` when you run it. It also usually gives the review task to the `Review Team` where the hosted model picks `Worker A`. The prompt's "reasonable default" allows both, so the check accepts both.

In [5]:
# The same project manager on a local model served by Ollama.
# Prerequisites:
#   ollama pull qwen2.5:7b
# Run cell 4 first (the shared cell under "# LangChain"); this cell needs nothing else: not the OpenAI cell, and no OpenAI key.
#
# This model sends all four tool calls in one turn, guessing the ID that create_new_task will
# return, and they run concurrently, so an update can reach the task manager before the task
# exists. Telling it to call one tool at a time worked 3 runs in 5. This middleware keeps only
# the first call of each turn, so the model sees the real ID before it makes the next one.
from langchain.agents.middleware import after_model
from langchain_ollama import ChatOllama


@after_model
def one_tool_call_per_turn(state, runtime):
    last = state["messages"][-1]
    if len(last.tool_calls) > 1:
        return {"messages": [last.model_copy(update={"tool_calls": last.tool_calls[:1]})]}
    return None


ollama_llm = ChatOllama(temperature=0.5, model="qwen2.5:7b")

await run_simulation(build_pm_agent(ollama_llm, middleware=(one_tool_call_per_turn,)))

--- Project Manager Simulation ---

[User Request] I need a new login system implemented ASAP. It should be assigned to Worker B.


DEBUG: Task created - TASK-001: Implement a new login system


DEBUG: Task TASK-001 updated with {'priority': 'P0'}


DEBUG: Task TASK-001 updated with {'assigned_to': 'Worker B'}


--- The task to implement a new login system has been created with the following details:
- Task ID: TASK-001
- Description: Implement a new login system
- Priority: P0 (urgent)
- Assigned to: Worker B

The task is now listed and ready for action.

------------------------------------------------------------

[User Request] We need to review the marketing website content.


DEBUG: Task created - TASK-002: Review marketing website content


DEBUG: Task TASK-002 updated with {'priority': 'P1'}


DEBUG: Task TASK-002 updated with {'assigned_to': 'Review Team'}


--- The tasks are now listed and managed as follows:
- Task ID: TASK-001, Description: Implement a new login system, Priority: P0, Assigned To: Worker B
- Task ID: TASK-002, Description: Review marketing website content, Priority: P1, Assigned To: Review Team

These tasks are now ready for execution.

--- Simulation Complete ---

Actual board: {'TASK-001': ('P0', 'Worker B'), 'TASK-002': ('P1', 'Review Team')}
TASK-001 is P0 on Worker B: True
TASK-002 has a priority and an available worker: True
